In [ ]:
import os

# Phải đặt trước khi torch khởi tạo CUDA.
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from pathlib import Path
import time

import torch

# Cố định kết quả giữa các lần chạy: cùng seed -> cùng file nộp.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

ROOT = Path('/home/user/TACVU2')
DATA = ROOT / 'data'
TRAIN_DIR = DATA / 'training_set'
RUNS = ROOT / 'runs'
RUNS.mkdir(parents=True, exist_ok=True)
SPLIT = 'public_test'  # Đổi thành 'private_test' trong phase private.

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[cấu hình] ROOT={ROOT} | SPLIT={SPLIT}')
print('[cấu hình] thiết bị: ' + (f'{DEVICE} ({torch.cuda.get_device_name(0)})'
      if DEVICE.type == 'cuda' else str(DEVICE)))


In [ ]:
import json
from pathlib import Path
from PIL import Image
from IPython.display import display

rows = [
    json.loads(line)
    for line in (TRAIN_DIR / "manifest.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

target = "train-00460_p01"

for row in rows:
    for image_path in row["image_paths"]:
        if Path(image_path).stem == target:
            print("ID:", row["id"])
            print("Image:", image_path)
            print("Label:", row["label_path"])

            display(Image.open(TRAIN_DIR / image_path))
            print((TRAIN_DIR / row["label_path"]).read_text(encoding="utf-8"))

            break


In [ ]:
"""Minimal file/Markdown helpers for the Exam A student starter."""

import json
from pathlib import Path
import random
import unicodedata
import zipfile

import numpy as np


SEED = 2026080701


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)


def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def load_manifest(split_dir: Path) -> list[dict]:
    path = split_dir / "manifest.jsonl"
    if not path.is_file():
        raise FileNotFoundError(f"Không thấy manifest: {path}")
    return load_jsonl(path)


def read_label(split_dir: Path, record: dict) -> str:
    path = split_dir / record["label_path"]
    return unicodedata.normalize("NFC", path.read_text(encoding="utf-8")).strip() + "\n"


def split_markdown_row(line: str) -> list[str]:
    if not (line.startswith("|") and line.endswith("|")):
        raise ValueError("Dòng Markdown không bắt đầu/kết thúc bằng dấu |")
    cells: list[str] = []
    current: list[str] = []
    escaped = False
    for character in line[1:-1]:
        if escaped:
            current.append("\\" + character)
            escaped = False
        elif character == "\\":
            escaped = True
        elif character == "|":
            cells.append("".join(current).strip())
            current = []
        else:
            current.append(character)
    if escaped:
        current.append("\\")
    cells.append("".join(current).strip())
    return cells


def parse_markdown(markdown: str) -> list[list[list[str]]]:
    tables: list[list[list[str]]] = []
    for block in unicodedata.normalize("NFC", markdown).strip().split("\n\n"):
        lines = [line.strip() for line in block.splitlines() if line.strip()]
        if len(lines) < 3:
            raise ValueError("Bảng có ít hơn ba dòng")
        rows = [split_markdown_row(line) for line in lines]
        width = len(rows[0])
        if width < 2 or any(len(row) != width for row in rows):
            raise ValueError("Số ô giữa các dòng không nhất quán")
        if any(cell != "---" for cell in rows[1]):
            raise ValueError("Thiếu dòng phân cách Markdown")
        tables.append([rows[0], *rows[2:]])
    if not tables:
        raise ValueError("Không có bảng")
    return tables


def is_valid_markdown(markdown: str) -> bool:
    try:
        parse_markdown(markdown)
        return "```" not in markdown
    except (ValueError, IndexError):
        return False


def write_predictions(output_dir: Path, records: list[dict], markdowns: list[str]) -> None:
    if len(records) != len(markdowns):
        raise ValueError("Số prediction không khớp manifest")
    output_dir.mkdir(parents=True, exist_ok=True)
    expected = {f"{record['id']}.md" for record in records}
    for existing in output_dir.glob("*.md"):
        if existing.name not in expected:
            existing.unlink()
    for record, markdown in zip(records, markdowns, strict=True):
        normalized = unicodedata.normalize("NFC", markdown).strip() + "\n"
        if not is_valid_markdown(normalized):
            raise ValueError(f"Prediction không hợp lệ: {record['id']}")
        (output_dir / f"{record['id']}.md").write_text(normalized, encoding="utf-8")


def make_predictions_zip(output_dir: Path, zip_path: Path) -> None:
    files = sorted(output_dir.glob("*.md"))
    if not files:
        raise ValueError("Thư mục prediction rỗng")
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, path.name)   # tệp .md phải nằm ngay gốc ZIP


In [ ]:
"""OpenCV-only table structure helpers for Exam A.

The module contains no OCR model and reads no labels at inference time.  It is
deliberately conservative: reliable bordered tables are reconstructed; harder
borderless pages fall back to a small valid table in the public starter.
"""

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np


@dataclass(frozen=True)
class GridTable:
    bbox: tuple[int, int, int, int]
    x_edges: tuple[int, ...]
    y_edges: tuple[int, ...]

    @property
    def rows(self) -> int:
        return max(0, len(self.y_edges) - 1)

    @property
    def cols(self) -> int:
        return max(0, len(self.x_edges) - 1)


def _runs(mask: np.ndarray, minimum: int = 1, gap: int = 2) -> list[tuple[int, int]]:
    indices = np.flatnonzero(mask)
    if indices.size == 0:
        return []
    result: list[tuple[int, int]] = []
    start = previous = int(indices[0])
    for raw in indices[1:]:
        value = int(raw)
        if value - previous > gap:
            if previous - start + 1 >= minimum:
                result.append((start, previous))
            start = value
        previous = value
    if previous - start + 1 >= minimum:
        result.append((start, previous))
    return result


def _centres(runs: list[tuple[int, int]]) -> list[int]:
    return [round((left + right) / 2) for left, right in runs]


def load_gray(path: Path) -> np.ndarray:
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"Không đọc được ảnh: {path}")
    return gray


def deskew_gray(gray: np.ndarray, maximum_degrees: float = 4.0) -> tuple[np.ndarray, float]:
    """Estimate document rotation from long rules and return an axis-aligned page."""
    edges = cv2.Canny(gray, 60, 180)
    height, width = gray.shape
    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 1800,
        threshold=max(70, width // 12),
        minLineLength=max(180, width // 5),
        maxLineGap=max(12, width // 80),
    )
    angles: list[float] = []
    if lines is not None:
        for raw in np.asarray(lines).reshape(-1, 4):   # OpenCV 4 trả (N,1,4), OpenCV 5 trả (N,4)
            x0, y0, x1, y1 = map(float, raw)
            angle = float(np.degrees(np.arctan2(y1 - y0, x1 - x0)))
            while angle > 90:
                angle -= 180
            while angle < -90:
                angle += 180
            if abs(angle) <= maximum_degrees:
                angles.append(angle)
    if not angles:
        return gray, 0.0
    angle = float(np.median(angles))
    if abs(angle) < 0.08:
        return gray, 0.0
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    rotated = cv2.warpAffine(
        gray,
        matrix,
        (width, height),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )
    return rotated, angle


def binarize(gray: np.ndarray) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    return cv2.threshold(clahe, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]


def line_masks(binary: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    height, width = binary.shape
    horizontal = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (max(45, width // 18), 1)),
    )
    vertical = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(35, height // 45))),
    )
    return horizontal, vertical


def _line_positions(mask: np.ndarray, axis: int, extent: int) -> list[int]:
    projection = np.count_nonzero(mask, axis=axis)
    threshold = max(8, round(extent * 0.42))
    positions: list[int] = []
    for start, end in _runs(projection >= threshold, gap=3):
        # A dark title band is seen by the line morphology as one thick run.
        # Its two sides are row boundaries; a normal 1–3 px rule contributes a
        # single centre coordinate.
        if end - start >= 9:
            positions.extend((start, end))
        else:
            positions.append(round((start + end) / 2))
    return positions


def _dedupe(values: list[int], tolerance: int = 6) -> list[int]:
    if not values:
        return []
    groups = [[values[0]]]
    for value in values[1:]:
        if value - groups[-1][-1] <= tolerance:
            groups[-1].append(value)
        else:
            groups.append([value])
    return [round(sum(group) / len(group)) for group in groups]


def detect_grid_tables(gray: np.ndarray) -> list[GridTable]:
    binary = binarize(gray)
    horizontal, vertical = line_masks(binary)
    joined = cv2.dilate(
        cv2.bitwise_or(horizontal, vertical),
        cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)),
        iterations=1,
    )
    contours, _ = cv2.findContours(joined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = gray.shape
    candidates: list[GridTable] = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w < width * 0.48 or h < 80 or w * h < width * height * 0.012:
            continue
        pad = 4
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(width, x + w + pad), min(height, y + h + pad)
        local_h = horizontal[y0:y1, x0:x1]
        local_v = vertical[y0:y1, x0:x1]
        xs = _line_positions(local_v, axis=0, extent=y1 - y0)
        ys = _line_positions(local_h, axis=1, extent=x1 - x0)
        xs = _dedupe([x0 + value for value in xs])
        ys = _dedupe([y0 + value for value in ys])
        if len(xs) < 3 or len(ys) < 3:
            continue
        # Lines at the table boundary are occasionally one pixel shorter.  Use
        # the contour bounds only when they agree with the internal grid.
        if abs(xs[0] - x) > 10:
            xs.insert(0, x)
        if abs(xs[-1] - (x + w - 1)) > 10:
            xs.append(x + w - 1)
        if abs(ys[0] - y) > 10:
            ys.insert(0, y)
        if abs(ys[-1] - (y + h - 1)) > 10:
            ys.append(y + h - 1)
        if 2 <= len(xs) - 1 <= 18 and 2 <= len(ys) - 1 <= 60:
            candidates.append(GridTable((x, y, x + w, y + h), tuple(xs), tuple(ys)))

    # A merged grid may create nested contours; keep the largest region when
    # two boxes substantially overlap.
    candidates.sort(key=lambda item: (item.bbox[1], item.bbox[0], -item.rows * item.cols))
    kept: list[GridTable] = []
    for table in candidates:
        x0, y0, x1, y1 = table.bbox
        area = max(1, (x1 - x0) * (y1 - y0))
        duplicate = False
        for previous in kept:
            a0, b0, a1, b1 = previous.bbox
            intersection = max(0, min(x1, a1) - max(x0, a0)) * max(0, min(y1, b1) - max(y0, b0))
            if intersection / min(area, max(1, (a1 - a0) * (b1 - b0))) > 0.72:
                duplicate = True
                break
        if not duplicate:
            kept.append(table)
    return kept


def blank_markdown(tables: list[GridTable]) -> str:
    blocks: list[str] = []
    for table in tables:
        rows = max(2, table.rows)
        cols = max(2, table.cols)
        lines = ["| " + " | ".join([""] * cols) + " |"]
        lines.append("| " + " | ".join(["---"] * cols) + " |")
        for _ in range(rows - 1):
            lines.append("| " + " | ".join([""] * cols) + " |")
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks).strip() + "\n"


In [ ]:
# Bộ nhận dạng ô của baseline: cắt ô theo lưới đã dò được rồi đọc chữ
# bằng một CRNN huấn luyện từ đầu trên tập train.

from dataclasses import dataclass
from pathlib import Path
import random
import re
import unicodedata

import cv2
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset



MODEL_SEED = 2026081101
BOLD_RE = re.compile(r"^\*\*(.*)\*\*$", re.DOTALL)
EMPTY_TABLE = "| A | B |\n| --- | --- |\n| x | y |\n"
MAX_STARTER_COLUMNS = 7
MAX_STARTER_ROWS = 15


def clean_label_cell(value: str) -> str:
    match = BOLD_RE.match(value)
    if match:
        value = match.group(1)
    value = value.replace("<br>", "\n").replace("\\|", "|")
    return unicodedata.normalize("NFC", value)


def anchor_spans(table: list[list[str]]) -> dict[tuple[int, int], tuple[int, int]]:
    """Read merge spans from training labels only, for aligned OCR crops."""
    rows, cols = len(table), len(table[0])
    spans: dict[tuple[int, int], tuple[int, int]] = {}
    for row in range(rows):
        for col in range(cols):
            if table[row][col] in {"[[H]]", "[[V]]"}:
                continue
            col_span = 1
            while col + col_span < cols and table[row][col + col_span] == "[[H]]":
                col_span += 1
            row_span = 1
            while row + row_span < rows and table[row + row_span][col] == "[[V]]":
                row_span += 1
            spans[(row, col)] = (row_span, col_span)
    return spans


def normalize_crop(crop: np.ndarray, target_height: int = 40, max_width: int = 384) -> np.ndarray:
    if crop.size == 0:
        return np.full((target_height, 16), 255, dtype=np.uint8)
    if crop.ndim == 3:
        crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    crop = cv2.GaussianBlur(crop, (3, 3), 0)
    ink = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    border = max(2, round(min(crop.shape) * 0.04))
    ink[:border] = 0
    ink[-border:] = 0
    ink[:, :border] = 0
    ink[:, -border:] = 0
    ys, xs = np.nonzero(ink)
    if not len(xs):
        return np.full((target_height, 16), 255, dtype=np.uint8)
    y0, y1 = max(0, int(ys.min()) - 2), min(crop.shape[0], int(ys.max()) + 3)
    x0, x1 = max(0, int(xs.min()) - 3), min(crop.shape[1], int(xs.max()) + 4)
    trimmed = crop[y0:y1, x0:x1]
    scale = target_height / max(1, trimmed.shape[0])
    width = max(4, min(max_width, round(trimmed.shape[1] * scale)))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    return cv2.resize(trimmed, (width, target_height), interpolation=interpolation)


def encode_png(image: np.ndarray) -> bytes:
    success, data = cv2.imencode(".png", image)
    if not success:
        raise ValueError("Không mã hóa được crop")
    return data.tobytes()


def decode_png(data: bytes) -> np.ndarray:
    image = cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Không giải mã được crop")
    return image


@dataclass
class CellSample:
    image: bytes
    text: str


def collect_cell_samples(train_dir: Path, records: list[dict], limit: int) -> list[CellSample]:
    samples: list[CellSample] = []
    rng = random.Random(MODEL_SEED)
    # Small experiments stop early; the official budget keeps the historical
    # full-pool selection so the BTC reference checkpoint remains comparable.
    fast_sampling = 0 < limit and limit != 42000
    ordered_records = list(records)
    if fast_sampling:
        rng.shuffle(ordered_records)
    for record in ordered_records:
        for image_rel, label_rel in zip(record["image_paths"], record["page_label_paths"], strict=True):
            gray, _ = deskew_gray(load_gray(train_dir / image_rel))
            detected = detect_grid_tables(gray)
            labels = parse_markdown((train_dir / label_rel).read_text(encoding="utf-8"))
            if [(item.rows, item.cols) for item in detected] != [(len(table), len(table[0])) for table in labels]:
                continue
            for grid, label in zip(detected, labels, strict=True):
                for (row, col), (row_span, col_span) in anchor_spans(label).items():
                    text = clean_label_cell(label[row][col])
                    if not text or "\n" in text or len(text) > 52:
                        continue
                    x0, x1 = grid.x_edges[col] + 2, grid.x_edges[col + col_span] - 2
                    y0, y1 = grid.y_edges[row] + 2, grid.y_edges[row + row_span] - 2
                    samples.append(CellSample(encode_png(normalize_crop(gray[y0:y1, x0:x1])), text))
                    if fast_sampling and limit <= len(samples):
                        rng.shuffle(samples)
                        return samples[:limit]
    rng.shuffle(samples)
    return samples[:limit] if limit > 0 else samples


class Alphabet:
    def __init__(self, characters: list[str]):
        self.characters = characters
        self.to_id = {character: index + 1 for index, character in enumerate(characters)}

    @classmethod
    def fit(cls, texts: list[str]) -> "Alphabet":
        return cls(sorted({character for text in texts for character in text}))

    def encode(self, text: str) -> list[int]:
        return [self.to_id[character] for character in text if character in self.to_id]

    def decode(self, indices: list[int]) -> str:
        result: list[str] = []
        previous = -1
        for index in indices:
            if index and index != previous and index <= len(self.characters):
                result.append(self.characters[index - 1])
            previous = index
        return "".join(result)


class CellDataset(Dataset):
    def __init__(self, samples: list[CellSample], alphabet: Alphabet):
        self.samples = samples
        self.alphabet = alphabet

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = decode_png(sample.image).astype(np.float32) / 255.0
        image = torch.from_numpy(1.0 - image).unsqueeze(0)
        target = torch.tensor(self.alphabet.encode(sample.text), dtype=torch.long)
        return image, target


def collate_cells(batch):
    max_width = max(image.shape[-1] for image, _ in batch)
    images = torch.zeros((len(batch), 1, 40, max_width), dtype=torch.float32)
    targets, lengths, widths = [], [], []
    for row, (image, target) in enumerate(batch):
        images[row, :, :, : image.shape[-1]] = image
        targets.append(target)
        lengths.append(len(target))
        widths.append(image.shape[-1])
    return images, torch.cat(targets), torch.tensor(lengths), torch.tensor(widths)


class GridCRNN(nn.Module):
    """The unchanged random-initialized GridCRNN architecture."""

    def __init__(self, classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 48, 3, padding=1), nn.BatchNorm2d(48), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(48, 96, 3, padding=1), nn.BatchNorm2d(96), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(96, 192, 3, padding=1), nn.BatchNorm2d(192), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(192, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
        )
        self.sequence = nn.GRU(256, 192, num_layers=2, bidirectional=True, batch_first=True, dropout=0.15)
        self.classifier = nn.Linear(384, classes)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.features(images).mean(dim=2).transpose(1, 2)
        sequence, _ = self.sequence(features)
        return self.classifier(sequence).log_softmax(dim=-1)


def load_recognizer(model_dir: Path, device: torch.device):
    payload = torch.load(model_dir / "grid_crnn.pt", map_location="cpu", weights_only=True)
    alphabet = Alphabet(payload["characters"])
    model = GridCRNN(len(alphabet.characters) + 1)
    model.load_state_dict(payload["state_dict"])
    return model.to(device).eval(), alphabet


@torch.inference_mode()
def recognize_images(model, alphabet: Alphabet, images: list[np.ndarray], device: torch.device) -> list[str]:
    if not images:
        return []
    width = max(image.shape[1] for image in images)
    batch = torch.zeros((len(images), 1, 40, width), dtype=torch.float32)
    for row, image in enumerate(images):
        batch[row, 0, :, : image.shape[1]] = torch.from_numpy(1.0 - image.astype(np.float32) / 255.0)
    best_ids = model(batch.to(device)).argmax(dim=-1).cpu().tolist()
    return [alphabet.decode(indices) for indices in best_ids]


def grid_page_markdown(gray: np.ndarray, tables: list[GridTable], model, alphabet, device) -> str:
    """Direct one-cell-per-grid reconstruction; no inferred merges or templates."""
    blocks: list[str] = []
    for table in tables:
        locations: list[tuple[int, int]] = []
        crops: list[np.ndarray] = []
        for row in range(table.rows):
            for col in range(table.cols):
                x0, x1 = table.x_edges[col] + 2, table.x_edges[col + 1] - 2
                y0, y1 = table.y_edges[row] + 2, table.y_edges[row + 1] - 2
                locations.append((row, col))
                crops.append(normalize_crop(gray[y0:y1, x0:x1]))
        cells = [["" for _ in range(table.cols)] for _ in range(table.rows)]
        for (row, col), text in zip(locations, recognize_images(model, alphabet, crops, device), strict=True):
            # The starter reconstructs content only.  Detecting bold headers
            # and totals is part of the contestant extension.
            cells[row][col] = text.replace("|", "\\|")
        lines = ["| " + " | ".join(cells[0]) + " |"]
        lines.append("| " + " | ".join(["---"] * table.cols) + " |")
        lines.extend("| " + " | ".join(row) + " |" for row in cells[1:])
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks).strip() + "\n"


In [ ]:
MODEL_DIR = RUNS / 'model_a'
EPOCHS = 8
BATCH_SIZE = 48
LEARNING_RATE = 8e-4
MAX_SAMPLES = 1000

set_seed(SEED)
torch.manual_seed(SEED)
print(f'[huấn luyện] seed={SEED} | epochs={EPOCHS} | batch={BATCH_SIZE} | max_samples={MAX_SAMPLES}')

train_records = load_manifest(TRAIN_DIR)
samples = collect_cell_samples(TRAIN_DIR, train_records, MAX_SAMPLES)
if not samples:
    raise ValueError('Không trích được ô nào từ tập train')
print(f'[huấn luyện] {len(train_records)} tài liệu -> {len(samples)} ô dùng để học')

alphabet = Alphabet.fit([sample.text for sample in samples])
dataset = CellDataset(samples, alphabet)
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_cells,
    generator=torch.Generator().manual_seed(MODEL_SEED),
)
print(f'[huấn luyện] bảng chữ cái: {len(alphabet.characters)} ký tự')

model = GridCRNN(len(alphabet.characters) + 1).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
print(f'[huấn luyện] tham số mô hình: {sum(p.numel() for p in model.parameters()):,}')

model.train()
for epoch in range(1, EPOCHS + 1):
    total = 0.0
    count = 0
    started = time.time()
    for images, targets, target_lengths, widths in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = model(images)
        input_lengths = torch.clamp((widths + 3) // 4, max=logits.shape[1]).to(dtype=torch.long)
        loss = criterion(logits.transpose(0, 1), targets, input_lengths, target_lengths)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += float(loss.detach().cpu()) * len(images)
        count += len(images)
    print(f'[huấn luyện] epoch {epoch}/{EPOCHS} | loss={total / max(1, count):.5f} '
          f'| {time.time() - started:.1f}s')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
torch.save({
    'state_dict': model.state_dict(),
    'characters': alphabet.characters,
    'seed': MODEL_SEED,
    'samples': len(samples),
    'epochs': EPOCHS,
}, MODEL_DIR / 'grid_crnn.pt')
print(f"[huấn luyện] đã lưu {MODEL_DIR / 'grid_crnn.pt'}")


In [ ]:
SPLIT_DIR = DATA / SPLIT
PREDICTIONS_DIR = RUNS / f'predictions_{SPLIT}'

set_seed(MODEL_SEED)
recognizer, recognizer_alphabet = load_recognizer(MODEL_DIR, DEVICE)
records = load_manifest(SPLIT_DIR)
print(f'[dự đoán] {len(records)} tài liệu trong {SPLIT}')

outputs = []
grid_pages = 0
started = time.time()

for record in records:
    page_outputs = []
    for relative in record['image_paths']:
        gray, _ = deskew_gray(load_gray(SPLIT_DIR / relative))
        # Baseline chỉ dựng lại bảng kẻ viền đầy đủ và có kích thước vừa phải.
        tables = [
            table for table in detect_grid_tables(gray)
            if 2 <= table.cols <= MAX_STARTER_COLUMNS
            and 2 <= table.rows <= MAX_STARTER_ROWS
        ]
        if tables:
            page_outputs.append(
                grid_page_markdown(gray, tables, recognizer, recognizer_alphabet, DEVICE).strip()
            )
            grid_pages += 1
    outputs.append(('\n\n'.join(page_outputs).strip() + '\n') if page_outputs else EMPTY_TABLE)

write_predictions(PREDICTIONS_DIR, records, outputs)
print(f'[dự đoán] {len(outputs)} tệp Markdown | {grid_pages} trang dò được lưới '
      f'| {time.time() - started:.1f}s')


In [ ]:
SUBMISSION_PATH = RUNS / f'submission_{SPLIT}.zip'

make_predictions_zip(PREDICTIONS_DIR, SUBMISSION_PATH)
print(f'[nộp bài] đã tạo {SUBMISSION_PATH}')
